# Workstream 3: Efficiency and baseline comparisons
Thin driver around `cap6411_atm.benchmark`. Measures latency/peak memory of the
DINOv3 BraTS baseline and the reduced-input-resolution baseline, and (once data
access is available) their validation Dice.

Rules for numbers that go into the report:
- **Same GPU for every compared variant.** Reported numbers come from one machine
  (RTX 5090); Colab runs are smoke tests only. Every CSV is saved with an env JSON.
- **Measured vs theoretical:** only `latency_*` and `*_memory_mb` columns are
  measurements; `tokens_theoretical` is not a speedup.
- **Validation split only.** The isolated test split is never loaded here.
- `random (timing only)` runs are valid for latency, never for Dice.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "cap6411_atm").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
sys.path.insert(0, str(REPO_ROOT / "src"))

from cap6411_atm import PROJECT_ROOT
assert PROJECT_ROOT == REPO_ROOT
print(f"Project root: {PROJECT_ROOT}")

## 1. Configure
Leave `CHECKPOINT` as `None` to benchmark the official ViT-B/16 architecture with
random weights (latency does not depend on weight values). Set `CHECKPOINT` and
`ARTIFACT` only with your own authorized DINOv3 access and the restricted
Workstream 1 adapter/decoder package.

In [ ]:
import torch

DINOV3_REPO = "/path/to/dinov3"   # official checkout at revision 6876159a...
CHECKPOINT = None                 # e.g. ".../dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
ARTIFACT = None                   # e.g. ".../dinov3_vitb16_brats_epoch2_adapter_decoder.pt"
from cap6411_atm.benchmark import load_benchmark_config

# Shared settings (resolutions, batch sizes, warm-up/iterations, fp16, stages, seed),
# the same file the command-line tool uses. Edit the JSON, not this cell.
SETTINGS = load_benchmark_config(PROJECT_ROOT / "configs" / "benchmark_resolution_sweep.json")
RESOLUTIONS = tuple(SETTINGS["sizes"])
BATCH_SIZES = tuple(SETTINGS["batch_sizes"])
WARMUP, ITERS = SETTINGS["warmup"], SETTINGS["iters"]
OUT_DIR = PROJECT_ROOT / "outputs" / "benchmarks"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
from cap6411_atm.benchmark import load_timing_only_dinov3_vitb16
from cap6411_atm.models import (
    BraTSDINOv3Segmenter,
    build_pretrained_brats_dinov3,
    load_adapter_decoder_artifact,
)

if CHECKPOINT:
    model = build_pretrained_brats_dinov3(DINOV3_REPO, CHECKPOINT)
    WEIGHTS = "official"
    if ARTIFACT:
        load_adapter_decoder_artifact(model, ARTIFACT)
        WEIGHTS += " + workstream1 adapter/decoder"
else:
    model = BraTSDINOv3Segmenter(load_timing_only_dinov3_vitb16(DINOV3_REPO))
    WEIGHTS = "random (timing only)"
model = model.to(device).eval()
print(WEIGHTS)

## 2. Latency and peak memory: resolution sweep
`stages=True` also times adapter / encoder / decoder separately. Lower
resolution (and any token merging) only shrinks the encoder share; the decoder
always upsamples to 240x240.

In [ ]:
from datetime import datetime

from cap6411_atm.benchmark import collect_env, resolution_sweep, run_suite, write_env, write_rows

variants = resolution_sweep(model, RESOLUTIONS)
rows = run_suite(variants, batch_sizes=BATCH_SIZES, device=device,
                 warmup=WARMUP, iters=ITERS, amp=SETTINGS["amp"],
                 stages=SETTINGS["stages"], seed=SETTINGS["seed"])

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
env = collect_env(device, weights=WEIGHTS)
csv_path = write_rows(rows, OUT_DIR / f"{stamp}_resolution_sweep.csv")
write_env(OUT_DIR / f"{stamp}_resolution_sweep_env.json", env)
print(csv_path, env.get("gpu_name"))
for r in rows:
    print(f"{r['variant']:>7} bs={r['batch_size']} tokens={r['tokens_theoretical']:>3} "
          f"median={r['latency_median_ms']:7.3f} ms  p90={r['latency_p90_ms']:7.3f} ms  "
          f"enc={r.get('encoder_median_ms', float('nan')):6.3f}  "
          f"dec={r.get('decoder_median_ms', float('nan')):6.3f}  "
          f"peak={r['peak_memory_mb']:8.1f} MB")

## 3. Validation Dice per resolution (needs authorized data + trained weights)
Uses the Workstream 1 validation split, sampler, and `StreamingDice`, so the
`res240` row should reproduce macro Dice ~0.819. Set `PILOT_PATIENTS` to a small
number for a quick check; use `None` for report numbers. The scope (pilot vs full split) and patient
count are saved in the CSV next to every Dice value.

In [ ]:
from cap6411_atm.benchmark import attach_quality, evaluate_quality_many, validation_loader

RUN_QUALITY = False
ARCHIVE = "/path/to/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData.zip"
MANIFEST = "/path/to/brats2023_gli_split_seed42.json"
PILOT_PATIENTS = 4   # None = full 187-patient validation split

quality = {}
if RUN_QUALITY:
    assert CHECKPOINT and ARTIFACT, "Dice needs official weights and the trained adapter/decoder"
    loader = validation_loader(ARCHIVE, MANIFEST, "configs/brats2023_gli_inspection.json",
                               patients=PILOT_PATIENTS)
    # One pass over the data scores every variant (each slice is decoded once).
    quality = evaluate_quality_many(variants, loader, device=device)
    for name, result in quality.items():
        print(name, round(result["macro_dice"], 4), result["dice"], result["scope"])
    attach_quality(rows, quality)
    write_rows(rows, OUT_DIR / f"{stamp}_resolution_sweep_with_dice.csv")

## 4. Report tables, matched-latency comparison, and trade-off plot
Once Workstream 2 provides merged variants (same `[B,4,240,240] -> [B,3,240,240]`
interface, with `benchmark_family` / `benchmark_setting` / `token_count`
attributes), add them to `variants`, re-run sections 2-3, then pair each merger
with the closest-latency reduced-resolution run. Benchmark content-adaptive
variants on real MRI slices (`run_suite(..., inputs=slices)`).

In [ ]:
from cap6411_atm.benchmark import match_latency, matched_table, plot_tradeoff, results_table

caption = f"{WEIGHTS}; {env.get('gpu_name', env['device'])}; {rows[0]['precision']}"
report = [results_table(rows, batch_size=b, caption=caption) for b in BATCH_SIZES]
print(report[0])
if quality:
    pairs = match_latency(rows, batch_size=1)
    if pairs:  # only once Workstream 2 variants are in `variants`
        report.append(matched_table(pairs))
        print(report[-1])
    print(plot_tradeoff(rows, OUT_DIR / f"{stamp}_tradeoff_bs1.png",
                        batch_size=1, gpu_name=env.get("gpu_name")))
(OUT_DIR / f"{stamp}_report_tables.md").write_text("\n".join(report))
# For the CVPR report: results_table(rows, fmt="latex", caption=...)